# Prismatic Minimal-Cell Escape Test
One-cell exact test of the proposed all-$n$ polygonal-prism extension of the homological cube-mobility mechanism.

Run the code cell below. No uploads, mounts, or GPU are required.


In [ ]:
# ================================================================
# PRISMATIC MINIMAL-CELL ESCAPE TEST
# Single self-contained Google Colab / Python block
# CPU only. Requires numpy, scipy, sympy (standard on Colab).
#
# Tests a new candidate extension of the cube-mobility theorem:
#
#   c_{N,n}^{prism} = (-1)^(n-1) 2^(n-1) C(2n-2,n-1)
#                     / [ N (N^2-1)^(n-1) ]
#
# for the square-free direct completion channel of an n-gonal prism.
# Here n side faces are inserted between the two n-gonal end faces,
# so the cell-completion channel occurs at perturbative order n = F-2.
#
# The script independently checks:
#   1) the cube n=4 history sum (-160) from raw face combinatorics;
#   2) the triangular-prism n=3 history sum (27/2 in E0 units);
#   3) the Catalan/central-binomial identity for n=3..12;
#   4) the closed all-N coefficient family symbolically;
#   5) an exact reduced Feshbach model that automatically resums all
#      subset paths and converges to the predicted coefficient;
#   6) cubical and triangular-prismatic T^3 cellular chain complexes;
#   7) H^2 dimension b2=3 on both regulators;
#   8) the cell-hop operator is non-scalar on ker(B2), hence escapes
#      scalar + boundary-ideal operators on both regulators.
#
# IMPORTANT SCOPE:
# This DOES NOT yet prove that the prism completion channel is the
# first total mobility term of the full SU(N) Hamiltonian. To prove
# that, one must additionally show all lower-order prism kernels are
# scalar modulo the boundary ideal after every microscopic channel is
# included. This code attacks the new necessary structure directly.
# ================================================================

from fractions import Fraction
from functools import lru_cache
from itertools import permutations
from collections import Counter, defaultdict
from math import comb
import numpy as np
import scipy.sparse as sp
from scipy.linalg import null_space
import sympy as sy

# ---------------------------
# User knobs
# ---------------------------
N_TEST = 3
L_HODGE = 3             # L=3 is fast and already topologically nontrivial
MAX_PRISM_N = 12        # exact subset DP is O(n 2^n), so 12 is trivial
FESHBACH_NS = (3, 4, 6) # triangular prism, cube, hexagonal prism
FESHBACH_LAMBDAS = (0.08, 0.04, 0.02, 0.01)
TOL = 1e-9

# ---------------------------
# Gate collector
# ---------------------------
gates = []

def gate(name, ok, detail=""):
    gates.append((name, bool(ok), str(detail)))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f" :: {detail}" if detail else ""))

# ================================================================
# PART I — RAW CELL-BOUNDARY HISTORY ENUMERATION
# ================================================================

def ekey(a, b):
    return tuple(sorted((a, b)))

def face_edges(face):
    return [ekey(face[i], face[(i + 1) % len(face)]) for i in range(len(face))]

def boundary_edges(faces, selected):
    counts = Counter()
    for fi in selected:
        for e in face_edges(faces[fi]):
            counts[e] += 1
    return [e for e, m in counts.items() if m % 2 == 1]

def connected_edge_set(edges):
    if not edges:
        return False
    adj = defaultdict(set)
    for a, b in edges:
        adj[a].add(b)
        adj[b].add(a)
    start = edges[0][0]
    seen, stack = set(), [start]
    while stack:
        x = stack.pop()
        if x in seen:
            continue
        seen.add(x)
        stack.extend(adj[x] - seen)
    verts = {x for e in edges for x in e}
    return seen == verts

def is_single_cycle(edges):
    if not edges:
        return False
    adj = defaultdict(set)
    for a, b in edges:
        adj[a].add(b)
        adj[b].add(a)
    if any(len(nbrs) != 2 for nbrs in adj.values()):
        return False
    return connected_edge_set(edges)

def exact_history_sum(faces, initial, final):
    """
    Sum direct square-free histories in units N^{-r} E0^{-(r-1)}.
    E(L)/E0 = L/p0, with p0 = initial-face perimeter.
    """
    side = [i for i in range(len(faces)) if i not in (initial, final)]
    p0 = len(face_edges(faces[initial]))
    total = Fraction(0, 1)
    records = []
    invalid = 0

    for order in permutations(side):
        selected = {initial}
        perims = []
        ok = True

        for step, fi in enumerate(order):
            current_boundary = set(boundary_edges(faces, selected))
            shared = [e for e in face_edges(faces[fi]) if e in current_boundary]
            if not shared or not connected_edge_set(shared):
                ok = False
                break

            selected.add(fi)
            if step < len(order) - 1:
                bdy = boundary_edges(faces, selected)
                if not is_single_cycle(bdy):
                    ok = False
                    break
                perims.append(len(bdy))

        if not ok:
            invalid += 1
            continue

        w = Fraction(1, 1)
        for L in perims:
            # 1 / (1 - E(L)/E0) = 1 / (1 - L/p0)
            w *= Fraction(p0, p0 - L)
        total += w
        records.append((order, tuple(perims), w))

    return total, records, invalid

# Cube: bottom, top, four sides
v = lambda x, y, z: (x, y, z)
CUBE_FACES = [
    [v(0,0,0), v(1,0,0), v(1,1,0), v(0,1,0)],
    [v(0,0,1), v(0,1,1), v(1,1,1), v(1,0,1)],
    [v(0,0,0), v(0,0,1), v(1,0,1), v(1,0,0)],
    [v(1,0,0), v(1,0,1), v(1,1,1), v(1,1,0)],
    [v(1,1,0), v(1,1,1), v(0,1,1), v(0,1,0)],
    [v(0,1,0), v(0,1,1), v(0,0,1), v(0,0,0)],
]

# Triangular prism: bottom, top, three rectangular sides
A=(0,0,0); B=(1,0,0); C=(0,1,0)
Ap=(0,0,1); Bp=(1,0,1); Cp=(0,1,1)
TRI_PRISM_FACES = [
    [A, C, B],
    [Ap, Bp, Cp],
    [A, B, Bp, Ap],
    [B, C, Cp, Bp],
    [C, A, Ap, Cp],
]

print("\n" + "="*88)
print("PART I — RAW TEMPORAL HISTORY CHECKS")
print("="*88)

cube_total, cube_records, cube_invalid = exact_history_sum(CUBE_FACES, 0, 1)
tri_total, tri_records, tri_invalid = exact_history_sum(TRI_PRISM_FACES, 0, 1)

cube_classes = Counter((r[1], r[2]) for r in cube_records)
tri_classes = Counter((r[1], r[2]) for r in tri_records)

print("Cube classes:")
for k, m in sorted(cube_classes.items(), key=lambda x: str(x[0])):
    print("  ", k, "multiplicity", m)
print("Triangular-prism classes:")
for k, m in sorted(tri_classes.items(), key=lambda x: str(x[0])):
    print("  ", k, "multiplicity", m)

gate("cube: all 24 side-face orderings admissible", len(cube_records)==24 and cube_invalid==0)
gate("cube: exact normalized history sum = -160", cube_total == Fraction(-160,1), cube_total)
gate("cube: histories are 16x(6,6,6;-8) + 8x(6,8,6;-4)",
     cube_classes == Counter({((6,6,6), Fraction(-8,1)):16,
                              ((6,8,6), Fraction(-4,1)):8}), cube_classes)
gate("triangular prism: all 3! side-face orderings admissible", len(tri_records)==6 and tri_invalid==0)
gate("triangular prism: exact normalized history sum = 27/2", tri_total == Fraction(27,2), tri_total)
gate("triangular prism: all histories have (5,5) and weight 9/4",
     tri_classes == Counter({((5,5), Fraction(9,4)):6}), tri_classes)

# ================================================================
# PART II — CATALAN / CENTRAL-BINOMIAL PRISM FAMILY
# ================================================================

def cyclic_components(mask, n):
    """Number of selected runs on C_n for a nonempty proper subset."""
    full = (1 << n) - 1
    if mask == 0 or mask == full:
        return 0
    transitions = sum(((mask >> i) & 1) != ((mask >> ((i+1) % n)) & 1)
                      for i in range(n))
    return transitions // 2

@lru_cache(None)
def catalan(k):
    return Fraction(comb(2*k, k), k+1)

def run_lengths(mask, n):
    """Lengths of cyclic selected components for a proper subset."""
    if mask == 0:
        return []
    full = (1 << n) - 1
    if mask == full:
        return [n]
    # start immediately after a zero, so no run wraps in the scan
    start = next(i for i in range(n) if not ((mask >> i) & 1))
    out, cur = [], 0
    for t in range(1, n+1):
        i = (start + t) % n
        if (mask >> i) & 1:
            cur += 1
        elif cur:
            out.append(cur)
            cur = 0
    if cur:
        out.append(cur)
    return out

def subset_order_weight_table(n):
    """
    f[S] = sum over all orders building S of prod_{nonempty prefixes T subseteq S} 1/c(T).
    For proper S, conjectured/exact recurrence solution is product Catalan_{run length}.
    """
    full = (1 << n) - 1
    f = {0: Fraction(1,1)}
    for size in range(1, n):
        for mask in range(1, full):
            if mask.bit_count() != size:
                continue
            c = cyclic_components(mask, n)
            s = sum((f[mask ^ (1 << i)] for i in range(n) if (mask >> i) & 1), Fraction(0,1))
            f[mask] = s / c
    return f

def terminal_history_factor(n):
    full = (1 << n) - 1
    f = subset_order_weight_table(n)
    # no 1/c factor on the full set: the final insertion lands directly on target face
    A = sum((f[full ^ (1 << i)] for i in range(n)), Fraction(0,1))
    return A, f

print("\n" + "="*88)
print("PART II — ALL-n PRISM COMBINATORICS")
print("="*88)
print(" n | order | A_n(history) | central binomial | K_n numerator | status")
print("---+-------+--------------+------------------+---------------+-------")

all_catalan_ok = True
all_terminal_ok = True
rows = []
for n in range(3, MAX_PRISM_N + 1):
    A_n, f = terminal_history_factor(n)
    full = (1 << n) - 1

    local_ok = True
    for mask, val in f.items():
        if mask == 0:
            continue
        pred = Fraction(1,1)
        for ell in run_lengths(mask, n):
            pred *= catalan(ell)
        if val != pred:
            local_ok = False
            break
    all_catalan_ok &= local_ok

    central = Fraction(comb(2*n - 2, n - 1), 1)
    term_ok = (A_n == central)
    all_terminal_ok &= term_ok

    K = ((-1)**(n-1)) * (2**(n-1)) * int(A_n)
    rows.append((n, A_n, K))
    print(f"{n:2d} | {n:5d} | {str(A_n):>12} | {comb(2*n-2,n-1):16d} | {K:13d} | {'PASS' if local_ok and term_ok else 'FAIL'}")

gate("every proper prism-side subset factorizes into Catalan(run length) products",
     all_catalan_ok, f"checked n=3..{MAX_PRISM_N}")
gate("terminal weighted history sum A_n = C(2n-2,n-1)",
     all_terminal_ok, f"checked n=3..{MAX_PRISM_N}")

# Symbolic coefficient family
N, n_sym = sy.symbols('N n', integer=True, positive=True)

def prism_coeff_closed(n, Nsym=N):
    return sy.factor(((-1)**(n-1)) * 2**(n-1) * sy.binomial(2*n-2, n-1)
                     / (Nsym * (Nsym**2 - 1)**(n-1)))

c3 = prism_coeff_closed(3)
c4 = prism_coeff_closed(4)
c6 = prism_coeff_closed(6)

print("\nClosed family:")
print("  c_{N,n} = (-1)^(n-1) 2^(n-1) binom(2n-2,n-1) / [N (N^2-1)^(n-1)]")
print("  n=3 triangular prism:", c3)
print("  n=4 cube:             ", c4)
print("  n=6 hexagonal prism:  ", c6)

gate("n=4 reproduces exact cube coefficient -160/[N(N^2-1)^3]",
     sy.simplify(c4 + sy.Rational(160,1)/(N*(N**2-1)**3)) == 0, c4)
gate("n=3 predicts +24/[N(N^2-1)^2]",
     sy.simplify(c3 - sy.Rational(24,1)/(N*(N**2-1)**2)) == 0, c3)
gate("n=6 predicts -8064/[N(N^2-1)^5]",
     sy.simplify(c6 + sy.Rational(8064,1)/(N*(N**2-1)**5)) == 0, c6)

print("\nLarge-N scaling of the direct n-prism completion channel:")
print("  c_{N,n} ~ const * N^{-(2n-1)} = N^{-(2F-5)},  F=n+2 faces.")

# ================================================================
# PART III — INDEPENDENT REDUCED FESHBACH TEST
# ================================================================

def reduced_prism_hamiltonian(n, Ncolor, lam):
    """
    Exact reduced loop-subspace toy Hamiltonian for the square-free prism channel.
    Basis = subsets of inserted side faces.
    Endpoint masks 0 and full have end-face perimeter n.
    Proper mask S has perimeter n + 2*c(S).
    Single side-face deformation matrix element = lam/N.
    """
    dim = 1 << n
    full = dim - 1
    CF = (Ncolor*Ncolor - 1) / (2*Ncolor)
    E0 = n * CF / 2
    H = np.zeros((dim, dim), dtype=float)

    for mask in range(dim):
        if mask in (0, full):
            L = n
        else:
            L = n + 2*cyclic_components(mask, n)
        H[mask, mask] = L * CF / 2

    for mask in range(dim):
        for i in range(n):
            nxt = mask ^ (1 << i)
            if nxt > mask:
                H[mask, nxt] = H[nxt, mask] = lam / Ncolor
    return H, E0

def feshbach_offdiag(n, Ncolor, lam):
    H, E0 = reduced_prism_hamiltonian(n, Ncolor, lam)
    full = H.shape[0] - 1
    P = [0, full]
    Q = [i for i in range(H.shape[0]) if i not in P]
    HPP = H[np.ix_(P, P)]
    HPQ = H[np.ix_(P, Q)]
    HQQ = H[np.ix_(Q, Q)]
    RQ = np.linalg.inv(E0*np.eye(len(Q)) - HQQ)
    Heff = HPP + HPQ @ RQ @ HPQ.T
    return Heff[0,1]

print("\n" + "="*88)
print("PART III — REDUCED FESHBACH CONVERGENCE")
print("="*88)

feshbach_ok = True
for n in FESHBACH_NS:
    pred = float(prism_coeff_closed(n, sy.Integer(N_TEST)))
    print(f"\nn={n}, N={N_TEST}, predicted coefficient = {pred:+.12g}")
    ratios = []
    for lam in FESHBACH_LAMBDAS:
        off = feshbach_offdiag(n, N_TEST, lam)
        ratio = off / (lam**n)
        ratios.append(ratio)
        print(f"  lambda={lam:0.3f}  Heff[0,1]/lambda^{n} = {ratio:+.12g}  error={ratio-pred:+.3e}")
    # Require smallest-lambda result within a modest numerical tolerance.
    ok = abs(ratios[-1] - pred) < 2e-4
    feshbach_ok &= ok
    gate(f"reduced Feshbach n={n} converges to closed coefficient", ok,
         f"last={ratios[-1]:+.12g}, pred={pred:+.12g}")

# ================================================================
# PART IV — CELLULAR HODGE TEST ON TWO T^3 REGULATORS
# ================================================================

def cycle_B1(L):
    B = np.zeros((L, L), dtype=float)
    for e in range(L):
        a, b = e, (e+1) % L
        B[a, e] -= 1
        B[b, e] += 1
    return sp.csr_matrix(B)

def triangulated_torus_2d(Lx, Ly):
    verts = [(i,j) for i in range(Lx) for j in range(Ly)]
    vid = {v:i for i,v in enumerate(verts)}
    tris = []
    for i in range(Lx):
        for j in range(Ly):
            v00 = vid[(i,j)]
            v10 = vid[((i+1)%Lx, j)]
            v11 = vid[((i+1)%Lx, (j+1)%Ly)]
            v01 = vid[(i, (j+1)%Ly)]
            tris.append((v00, v10, v11))
            tris.append((v00, v11, v01))

    edge_id = {}
    for tri in tris:
        for a,b in zip(tri, tri[1:]+tri[:1]):
            key = tuple(sorted((a,b)))
            if key not in edge_id:
                edge_id[key] = len(edge_id)

    edges = [None]*len(edge_id)
    for key, idx in edge_id.items():
        edges[idx] = key

    n0, n1, n2 = len(verts), len(edges), len(tris)
    B1 = np.zeros((n0,n1), dtype=float)
    for e,(a,b) in enumerate(edges):
        B1[a,e] -= 1
        B1[b,e] += 1

    B2 = np.zeros((n1,n2), dtype=float)
    for t,tri in enumerate(tris):
        for a,b in zip(tri, tri[1:]+tri[:1]):
            key = tuple(sorted((a,b)))
            e = edge_id[key]
            B2[e,t] += +1 if (a,b) == key else -1

    return sp.csr_matrix(B1), sp.csr_matrix(B2), n0, n1, n2

def triangular_prism_torus(Lx, Ly, Lz):
    """Cell complex (triangulated T^2) x S^1: every 3-cell is a triangular prism."""
    B1b, B2b, n0, n1, n2 = triangulated_torus_2d(Lx, Ly)
    Bz = cycle_B1(Lz)
    Iz0 = sp.eye(Lz, format='csr')
    Iz1 = sp.eye(Lz, format='csr')

    # C2 = (base C2 x z C0)  +  (base C1 x z C1)
    # C1 = (base C1 x z C0)  +  (base C0 x z C1)
    top_left  = sp.kron(B2b, Iz0, format='csr')
    top_right = -sp.kron(sp.eye(n1, format='csr'), Bz, format='csr')
    bot_left  = sp.csr_matrix((n0*Lz, n2*Lz))
    bot_right = sp.kron(B1b, Iz1, format='csr')
    B2 = sp.bmat([[top_left, top_right], [bot_left, bot_right]], format='csr')

    # C3 = base C2 x z C1
    B3_top = sp.kron(sp.eye(n2, format='csr'), Bz, format='csr')
    B3_bot = sp.kron(B2b, Iz1, format='csr')
    B3 = sp.vstack([B3_top, B3_bot], format='csr')

    # Minimal-cell end-face hop: bottom triangle <-> top triangle across each prism.
    Az = np.zeros((Lz,Lz), dtype=float)
    for z in range(Lz):
        zp = (z+1) % Lz
        Az[z,zp] += 1
        Az[zp,z] += 1
    H_horizontal = sp.kron(sp.eye(n2, format='csr'), sp.csr_matrix(Az), format='csr')
    n_vertical_faces = n1*Lz
    H = sp.block_diag((H_horizontal,
                       sp.csr_matrix((n_vertical_faces,n_vertical_faces))), format='csr')

    n_prisms = n2*Lz
    return B2, B3, H, n_prisms

def cubical_torus(L):
    """Standard periodic cubic T^3 cell complex and opposite-face cube hop."""
    edges, eid = [], {}
    for d in range(3):
        for x in range(L):
            for y in range(L):
                for z in range(L):
                    eid[(d,x,y,z)] = len(edges)
                    edges.append((d,x,y,z))

    faces, fid = [], {}
    for o in range(3):
        for x in range(L):
            for y in range(L):
                for z in range(L):
                    fid[(o,x,y,z)] = len(faces)
                    faces.append((o,x,y,z))

    cubes = [(x,y,z) for x in range(L) for y in range(L) for z in range(L)]
    nE, nF, nC = len(edges), len(faces), len(cubes)
    B2 = np.zeros((nE,nF), dtype=float)

    def add_edge(fcol, d, x, y, z, s):
        B2[eid[(d,x%L,y%L,z%L)], fcol] += s

    for o,x,y,z in faces:
        f = fid[(o,x,y,z)]
        if o == 0: # xy
            add_edge(f,0,x,y,z,+1); add_edge(f,1,x+1,y,z,+1)
            add_edge(f,0,x,y+1,z,-1); add_edge(f,1,x,y,z,-1)
        elif o == 1: # xz
            add_edge(f,0,x,y,z,+1); add_edge(f,2,x+1,y,z,+1)
            add_edge(f,0,x,y,z+1,-1); add_edge(f,2,x,y,z,-1)
        else: # yz
            add_edge(f,1,x,y,z,+1); add_edge(f,2,x,y+1,z,+1)
            add_edge(f,1,x,y,z+1,-1); add_edge(f,2,x,y,z,-1)

    B3 = np.zeros((nF,nC), dtype=float)
    cid = {c:i for i,c in enumerate(cubes)}
    for x,y,z in cubes:
        c = cid[(x,y,z)]
        B3[fid[(0,x,y,(z+1)%L)],c] += +1
        B3[fid[(0,x,y,z)],c]       += -1
        B3[fid[(1,x,(y+1)%L,z)],c] += -1
        B3[fid[(1,x,y,z)],c]       += +1
        B3[fid[(2,(x+1)%L,y,z)],c] += +1
        B3[fid[(2,x,y,z)],c]       += -1

    H = np.zeros((nF,nF), dtype=float)
    for o,x,y,z in faces:
        i = fid[(o,x,y,z)]
        if o == 0:
            j = fid[(o,x,y,(z+1)%L)]
        elif o == 1:
            j = fid[(o,x,(y+1)%L,z)]
        else:
            j = fid[(o,(x+1)%L,y,z)]
        H[i,j] += 1
        H[j,i] += 1

    return sp.csr_matrix(B2), sp.csr_matrix(B3), sp.csr_matrix(H), nC

def analyze_hodge(name, B2, B3, Hcell, n3cells):
    chain = B2 @ B3
    chain_err = 0.0 if chain.nnz == 0 else float(np.max(np.abs(chain.data)))

    K = null_space(B2.toarray(), rcond=1e-10)
    ker_dim = K.shape[1]
    rank_B3 = np.linalg.matrix_rank(B3.toarray(), tol=1e-10)
    b2 = ker_dim - rank_B3

    ideal = B2.T @ B2
    ideal_comp = K.T @ (ideal @ K)
    ideal_norm = float(np.linalg.norm(ideal_comp))

    Hk = K.T @ (Hcell @ K)
    Hk = 0.5*(Hk + Hk.T)
    evals = np.linalg.eigvalsh(Hk)
    spread = float(np.ptp(evals))
    scalar_residual = float(np.linalg.norm(Hk - np.trace(Hk)/Hk.shape[0]*np.eye(Hk.shape[0])))

    print(f"\n{name}")
    print(f"  B2 shape={B2.shape}, B3 shape={B3.shape}")
    print(f"  ||B2 B3||_max             = {chain_err:.3e}")
    print(f"  dim ker(B2)               = {ker_dim}")
    print(f"  rank(B3)                  = {rank_B3}")
    print(f"  b2 = dim ker(B2)-rank(B3) = {b2}")
    print(f"  ||K^T (B2^T B2) K||       = {ideal_norm:.3e}")
    print(f"  cell-hop eigenvalue spread= {spread:.12g}")
    print(f"  non-scalar residual norm  = {scalar_residual:.12g}")

    gate(f"{name}: chain condition B2 B3 = 0", chain_err < TOL, chain_err)
    gate(f"{name}: torus H^2 has b2=3", b2 == 3, b2)
    gate(f"{name}: boundary ideal annihilates ker(B2)", ideal_norm < 1e-8, ideal_norm)
    gate(f"{name}: minimal-cell hop escapes scalar + boundary ideal",
         spread > 1e-6 and scalar_residual > 1e-6,
         f"spread={spread:.6g}, residual={scalar_residual:.6g}")
    gate(f"{name}: dim ker(B2)=#3cells+2 on T^3", ker_dim == n3cells + 2,
         f"ker={ker_dim}, cells+2={n3cells+2}")

    return dict(chain_err=chain_err, ker_dim=ker_dim, rank_B3=rank_B3,
                b2=b2, ideal_norm=ideal_norm, spread=spread,
                scalar_residual=scalar_residual)

print("\n" + "="*88)
print("PART IV — HODGE / BOUNDARY-IDEAL ESCAPE ON TWO REGULATORS")
print("="*88)

B2c, B3c, Hc, nCube = cubical_torus(L_HODGE)
cube_hodge = analyze_hodge("cubical T^3", B2c, B3c, Hc, nCube)

B2p, B3p, Hp, nPrism = triangular_prism_torus(L_HODGE, L_HODGE, L_HODGE)
prism_hodge = analyze_hodge("triangular-prismatic T^3", B2p, B3p, Hp, nPrism)

# ================================================================
# FINAL VERDICT
# ================================================================
print("\n" + "="*88)
print("FINAL GATE SUMMARY")
print("="*88)
passed = sum(ok for _,ok,_ in gates)
for i,(name,ok,detail) in enumerate(gates,1):
    print(f"{i:02d}. {'PASS' if ok else 'FAIL'} — {name}" + (f" :: {detail}" if detail else ""))
print("-"*88)
print(f"PASSED {passed}/{len(gates)} GATES")

if passed == len(gates):
    print("\nRESULT: SUPPORTED at the exact cell-completion + Hodge-escape level.")
    print("The new all-n prism history law survives every test in this notebook.")
    print("The triangular-prism cell hop is demonstrably nontrivial on the Hodge-flat/cycle sector,")
    print("and the reduced Feshbach model independently recovers its predicted third-order coefficient.")
    print("\nNEXT FALSIFICATION TARGET:")
    print("Build the COMPLETE triangular-prismatic SU(N) strong-coupling effective kernel through order 3.")
    print("If all order-1 and order-2 momentum-dependent pieces are scalar modulo the boundary ideal,")
    print("then the minimal-cell escape principle is confirmed on a genuinely non-cubic regulator.")
    print("If any lower-order projected shape survives, the 'first escape = F-2' conjecture is false.")
else:
    print("\nRESULT: AT LEAST ONE GATE FAILED. Treat the proposed prism law as falsified or the implementation as suspect.")



PART I — RAW TEMPORAL HISTORY CHECKS
Cube classes:
   ((6, 6, 6), Fraction(-8, 1)) multiplicity 16
   ((6, 8, 6), Fraction(-4, 1)) multiplicity 8
Triangular-prism classes:
   ((5, 5), Fraction(9, 4)) multiplicity 6
[PASS] cube: all 24 side-face orderings admissible
[PASS] cube: exact normalized history sum = -160 :: -160
[PASS] cube: histories are 16x(6,6,6;-8) + 8x(6,8,6;-4) :: Counter({((6, 6, 6), Fraction(-8, 1)): 16, ((6, 8, 6), Fraction(-4, 1)): 8})
[PASS] triangular prism: all 3! side-face orderings admissible
[PASS] triangular prism: exact normalized history sum = 27/2 :: 27/2
[PASS] triangular prism: all histories have (5,5) and weight 9/4 :: Counter({((5, 5), Fraction(9, 4)): 6})

PART II — ALL-n PRISM COMBINATORICS
 n | order | A_n(history) | central binomial | K_n numerator | status
---+-------+--------------+------------------+---------------+-------
 3 |     3 |            6 |                6 |            24 | PASS
 4 |     4 |           20 |               20 |          